In [ ]:
import requests
import time

payload = {
    "model": "qwen2.5-coder:7b",
    "prompt": "Explain graph databases in one paragraph.",
    "stream": False
}

response = requests.post("http://localhost:11434/api/generate", json=payload).json()

# Extract performance metadata provided by Ollama
output_tokens = response["eval_count"]
# Ollama records durations in nanoseconds; convert to seconds
generation_time_sec = response["eval_duration"] / 1_000_000_000 

tokens_per_second = output_tokens / generation_time_sec

print(f"Generated {output_tokens} tokens in {generation_time_sec:.2f} seconds.")
print(f"Actual Speed: {tokens_per_second:.2f} tokens/sec")

Generated 109 tokens in 3.90 seconds.
Actual Speed: 27.92 tokens/sec


In [20]:
%pip install langgraph

/Users/apple/dev/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
!uv pip install neo4j



Using Python 3.14.5 environment at: /Users/apple/dev/.venv
Checked 1 package in 16ms


In [22]:
!uv pip install -U langchain-experimental langchain-ollama


Using Python 3.14.5 environment at: /Users/apple/dev/.venv
Resolved 46 packages in 822ms                                        
Checked 46 packages in 1ms


In [23]:
!uv pip install -U langchain-groq

Using Python 3.14.5 environment at: /Users/apple/dev/.venv
Resolved 31 packages in 196ms                                        
Checked 31 packages in 0.64ms


In [1]:
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_experimental.graph_transformers import LLMGraphTransformer

load_dotenv()


/Users/apple/dev/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/cs/v8d5n4wj30z1p1w6v_30t_3h0000gn/T/ipykernel_59779/2495439992.py:3: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


True

In [ ]:
llm = ChatOllama(
    model="qwen2.5-coder:7b",
    base_url="http://127.0.0.1:11434",
    temperature=0.9,
)
response = llm.invoke("What is the capital of France?")
print(response.content)


The capital of France is Paris.


In [3]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import TokenTextSplitter
import time
# Read the wikipedia article

try:
    loader = WikipediaLoader(query="Chandragupta Maurya", load_max_docs=2)
    raw_documents = loader.load()
except Exception as e:
    print(f"Wikipedia load failed: {e}")
    print("Retrying in 5 seconds...")
    time.sleep(5)
    raw_documents = loader.load()
# Define chunking strategy
text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=24)
documents = text_splitter.split_documents(raw_documents[:3])

In [4]:
import os
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.graphs import Neo4jGraph

# llm = Ollama(
#     model="qwen-2.5-coder-7b",
#     base_url="http://127.0.0.1:11434",
#     temperature=0.9,
# )
llm_transformer = LLMGraphTransformer(llm=llm)

# Extract graph data
graph_documents = llm_transformer.convert_to_graph_documents(documents)

# Initialize Neo4j connection with credentials from .env
graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD")
)

# Store to neo4j
graph.add_graph_documents(
  graph_documents, 
  baseEntityLabel=True, 
  include_source=True
)


/var/folders/cs/v8d5n4wj30z1p1w6v_30t_3h0000gn/T/ipykernel_59779/2233432584.py:16: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=1, column=257, offset=256>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position'